# 13. Efficiency profile (no training, about 5 minutes)

Accuracy is not the only thing a model is judged on. This notebook measures, for the models in your project, on the same GPU:
- **parameters** (millions)
- **approximate compute** (GMACs for one 224 px image)
- **inference speed** (images per second, fp16, batch 32) and peak GPU memory
- **training speed** (images per second, fp16, batch 16) and peak GPU memory

It answers questions such as: does the hybrid use fewer parameters than a plain ResNet-50, and what does the Mamba part cost in speed?

**Read the numbers carefully.**
- The compute figure counts convolutions and matrix multiplications only. The Mamba scan is not counted, so Mamba-based models are **under**counted.
- The Mamba blocks here use a **pure-PyTorch scan**, much slower than the paper's CUDA kernels. Do not compare speeds with the paper's table.
- The models are randomly initialised, because speed and size do not depend on the weights, so no downloads are needed.

Before running: Accelerator = GPU T4, Internet ON (only for installing `mambapy`). No datasets are needed. Results file: `results_13.csv`.

## 1. Install

In [ ]:
!pip install -q mambapy

## 2. Settings

In [ ]:
import os, time, math, copy, numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision

DIM, DEPTH, HYB_DEPTH = 192, 8, 4
DEV = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEV == "cuda"
print("device:", DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else "")

## 3. Models

In [ ]:
from mambapy.mamba import MambaBlock, MambaConfig

def make_block(d):
    return MambaBlock(MambaConfig(d_model=d, n_layers=1, d_state=16, expand_factor=2,
                                  d_conv=4, pscan=True, use_cuda=False))

class BiMambaBlock(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.norm = nn.LayerNorm(d)
        self.fwd, self.bwd = make_block(d), make_block(d)
        self.wf = nn.Parameter(torch.tensor(0.5))   # learnable fusion weights (paper Eq. 7)
        self.wb = nn.Parameter(torch.tensor(0.5))
    def forward(self, x):
        h = self.norm(x)
        return x + self.wf * self.fwd(h) + self.wb * self.bwd(h.flip(1)).flip(1)

class MiniBiMamba(nn.Module):
    def __init__(self, d=192, depth=8, patch=16, base_grid=14):
        super().__init__()
        self.embed = nn.Conv2d(3, d, patch, patch)
        self.pos = nn.Parameter(torch.zeros(1, d, base_grid, base_grid))
        nn.init.trunc_normal_(self.pos, std=0.02)
        self.blocks = nn.ModuleList([BiMambaBlock(d) for _ in range(depth)])
        self.norm = nn.LayerNorm(d)
    def forward(self, img):
        t = self.embed(img)
        gh, gw = t.shape[-2:]
        pos = F.interpolate(self.pos, size=(gh, gw), mode="bicubic", align_corners=False)
        t = (t + pos).flatten(2).transpose(1, 2)
        for blk in self.blocks:
            t = blk(t)
        return self.norm(t), (gh, gw)

class Classifier(nn.Module):
    def __init__(self, n_classes, d=DIM, depth=DEPTH):
        super().__init__()
        self.enc = MiniBiMamba(d, depth)
        self.head = nn.Linear(d, n_classes)
    def forward(self, x):
        tokens, _ = self.enc(x)
        return self.head(tokens.mean(1))

_m = Classifier(28)
print("Parameters (M):", round(sum(p.numel() for p in _m.parameters()) / 1e6, 2)); del _m

In [ ]:
class ResNetTokenizer(nn.Module):
    """ImageNet-pretrained ResNet-50 up to layer3 (stride 16), then a 1x1 projection to the Mamba width.
    It replaces the 16x16 patch-embedding conv: a 224 px image still gives a 14x14 token grid,
    160 px gives 10x10 and 96 px gives 6x6, exactly like the PSMamba views."""
    def __init__(self, d):
        super().__init__()
        r = torchvision.models.resnet50(weights=None)   # needs Internet ON
        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu, r.maxpool, r.layer1, r.layer2, r.layer3)
        self.proj = nn.Conv2d(1024, d, 1)
    def forward(self, x):
        return self.proj(self.stem(x))                                  # (B, d, gh, gw)

class HybridMamba(nn.Module):
    def __init__(self, n_classes, d=DIM, depth=HYB_DEPTH, base_grid=14):
        super().__init__()
        self.tok = ResNetTokenizer(d)
        self.pos = nn.Parameter(torch.zeros(1, d, base_grid, base_grid))
        nn.init.trunc_normal_(self.pos, std=0.02)
        self.blocks = nn.ModuleList([BiMambaBlock(d) for _ in range(depth)])    # same Bi-Mamba blocks as before
        self.norm = nn.LayerNorm(d)
        self.head = nn.Linear(d, n_classes)
    def forward(self, img):
        t = self.tok(img)
        gh, gw = t.shape[-2:]
        pos = F.interpolate(self.pos, size=(gh, gw), mode="bicubic", align_corners=False)
        t = (t + pos).flatten(2).transpose(1, 2)
        for blk in self.blocks:
            t = blk(t)
        return self.head(self.norm(t).mean(1))

def run_hybrid(name, fold=0, seed=0, label="hybrid", epochs=None):
    ep_total = epochs or EPOCHS
    tag = f"{label}_{name}_f{fold}_s{seed}"
    if already_done(tag):
        print("skip (already done):", tag); return
    set_seed(seed)
    tr, va, te, cmap = get_split(name, fold)
    ltr, lva, lte = make_loaders(tr, va, te, cmap, bs=BATCH)
    n_cls = len(cmap)
    model = HybridMamba(n_cls).to(DEV)
    stem = list(model.tok.stem.parameters()); stem_ids = {id(p) for p in stem}
    rest = [p for p in model.parameters() if id(p) not in stem_ids]
    opt = torch.optim.AdamW([{"params": stem, "lr": LR_BACKBONE, "weight_decay": WD},
                             {"params": rest, "lr": LR, "weight_decay": WD}])
    total, warm = ep_total * len(ltr), WARMUP * len(ltr)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    best, t_start = dict(val_acc=-1.0, epoch=-1, state=None), time.time()
    for ep in range(ep_total):
        model.train(); t0, tl, nb = time.time(), 0.0, 0
        for x, y in ltr:
            x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                out = model(x)
            loss = F.cross_entropy(out.float(), y, label_smoothing=0.1)
            opt.zero_grad(set_to_none=True)
            if not torch.isfinite(loss):
                sched.step(); continue
            scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            tl += loss.item(); nb += 1
        r = evaluate(model, lva, n_cls)
        if r["acc"] > best["val_acc"]:
            best = dict(val_acc=r["acc"], epoch=ep, state={k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
        print(f"[{tag}] epoch {ep+1}/{ep_total} | train loss {tl/max(nb,1):.3f} | val acc {r['acc']:.2f} | best {best['val_acc']:.2f} (ep {best['epoch']+1}) | {time.time()-t0:.0f}s")
    model.load_state_dict(best["state"])
    res = evaluate(model, lte, n_cls)
    row = dict(tag=tag, label=label, dataset=name, fold=fold, seed=seed, epochs=ep_total, best_epoch=best["epoch"] + 1,
               val_acc=best["val_acc"], test_acc=res["acc"], test_macro_f1=res["macro_f1"],
               test_minority_f1=minority_f1(res, tr, cmap), minutes=(time.time() - t_start) / 60)
    pd.DataFrame([row]).to_csv(RESULTS, mode="a", header=not os.path.exists(RESULTS), index=False)
    print("RESULT", {k: (round(v, 2) if isinstance(v, float) else v) for k, v in row.items()})

## 4. Profile

In [ ]:
from torch.utils.flop_counter import FlopCounterMode

def sync():
    if DEV == "cuda": torch.cuda.synchronize()

def profile(name, model, n_cls=28, res=224, batch=32, iters=20):
    model = model.to(DEV)
    params = sum(p.numel() for p in model.parameters()) / 1e6
    row = dict(model=name, params_M=round(params, 2))
    # approximate compute (counts convolutions and matrix multiplications only; divide by 2 to get multiply-accumulates)
    try:
        model.eval()
        with FlopCounterMode(display=False) as fc, torch.no_grad():
            model(torch.randn(1, 3, res, res, device=DEV))
        row["GMACs_approx"] = round(fc.get_total_flops() / 2 / 1e9, 2)
    except Exception as e:
        row["GMACs_approx"] = float("nan")
    # inference throughput and memory (fp16)
    x = torch.randn(batch, 3, res, res, device=DEV)
    model.eval()
    if DEV == "cuda": torch.cuda.reset_peak_memory_stats()
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
        for _ in range(3): model(x)
        sync(); t0 = time.time()
        for _ in range(iters): model(x)
        sync()
    row["infer_img_per_s"] = round(batch * iters / (time.time() - t0), 1)
    row["infer_peak_GB"] = round(torch.cuda.max_memory_allocated() / 1e9, 2) if DEV == "cuda" else float("nan")
    # training step throughput (fp16, batch 16)
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=1e-5)
    xb, yb = torch.randn(16, 3, res, res, device=DEV), torch.randint(0, n_cls, (16,), device=DEV)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    if DEV == "cuda": torch.cuda.reset_peak_memory_stats()
    for i in range(8):
        if i == 3: sync(); t0 = time.time()
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            loss = F.cross_entropy(model(xb).float(), yb)
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    sync()
    row["train_img_per_s"] = round(16 * 5 / (time.time() - t0), 1)
    row["train_peak_GB"] = round(torch.cuda.max_memory_allocated() / 1e9, 2) if DEV == "cuda" else float("nan")
    del model, opt, x, xb
    if DEV == "cuda": torch.cuda.empty_cache()
    return row

builders = {
    "Bi-Mamba from scratch (baseline)": lambda: Classifier(28),
    "Hybrid (ResNet-50 to layer3 + 4 Bi-Mamba)": lambda: HybridMamba(28),
    "ResNet-50": lambda: torchvision.models.resnet50(weights=None, num_classes=28),
    "ConvNeXt-Tiny": lambda: torchvision.models.convnext_tiny(weights=None, num_classes=28),
    "Swin-T": lambda: torchvision.models.swin_t(weights=None, num_classes=28),
}
rows = []
for name, build in builders.items():
    try:
        rows.append(profile(name, build()))
        print("done:", name)
    except Exception as e:
        print("FAILED:", name, repr(e)[:200])
df = pd.DataFrame(rows)
df.to_csv("/kaggle/working/results_13.csv", index=False)
print()
print(df.to_string(index=False))